In [1]:
from datetime import date
import polars as pl

from flights_ai_model_airport.flights_ai_model_utils import get_latest_airport_model_path
from flights_ai_model_airport.inference import load_model, run_score_canidates, score_airport_candidates
from flights_ai_model_airport.train import get_training_flights
MODEL_PATH = get_latest_airport_model_path()

target_date = date(2026,3,15)
df_flights = get_training_flights(target_date)

Loading airports from /Volumes/T2-SSD/planequery/data/interim/airports.csv


In [2]:
df = score_airport_candidates(df_flights, load_model(MODEL_PATH))
df

Loading airport model from /Volumes/T2-SSD/planequery/data/models/flights_ai_model_airport/runs/2026-07-01_17-38_e6162c2_4days/model.pkl


/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


flight_id,airport_ident,score,distance_km,elevation_above_airport_ft,bearing_airport_to_first_deg,bearing_first_to_airport_deg,takeoff_track_error_deg,airport_behind_error_deg,along_track_from_airport_km,cross_track_from_airport_km,candidate_is_behind,rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""a0003a_2026-03-15_12-59""","""KAPF""",0.999844,1.357153,342.0,44.011419,224.015584,178.615582,178.611417,-1.356757,0.032789,0.0,1
"""a0003a_2026-03-15_12-59""","""FA37""",0.049956,8.067243,341.0,308.187051,128.159061,82.75906,82.787049,1.016812,8.002906,0.0,2
"""a0003a_2026-03-15_12-59""","""KMKY""",0.005005,20.716662,345.0,333.274324,153.233287,107.833286,107.874322,-6.344445,19.721261,0.0,3
"""a0003a_2026-03-15_12-59""","""1FD4""",0.000437,35.315802,321.0,206.620101,26.549843,18.850159,18.779901,33.421702,11.410335,1.0,4
"""a0003a_2026-03-15_12-59""","""US-5896""",0.000417,85.487856,330.0,218.647836,38.409459,6.990543,6.752165,84.852361,10.404343,1.0,5
…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf0ff_2026-03-15_01-59""","""US-2469""",0.000032,149.37861,0.0,200.321129,19.998966,146.901028,146.578865,-125.13872,81.573708,0.0,136
"""adf0ff_2026-03-15_01-59""","""US-2468""",0.000032,148.006378,0.0,202.844926,22.488224,144.41177,144.055068,-120.361795,86.133188,0.0,137
"""adf0ff_2026-03-15_01-59""","""US-4628""",0.000031,139.58386,0.0,201.054702,20.743625,146.156369,145.845292,-115.932855,77.738196,0.0,138


In [ ]:
# we'll do the main and get what flight_ids did not match for airports.

1. Get flights where model failed to match
2. Look at what the features were for those flight_ids

In [4]:
from evaluation.flights.evaluation import main
outputs,df_stats = main("trad-airport-model", "sfdps", test_dates = target_date, airport_model_path = MODEL_PATH, pia_or_american_ladd_only=True)
df_comp = outputs[0][0]


Loading airport model from /Volumes/T2-SSD/planequery/data/models/flights_ai_model_airport/runs/2026-07-01_17-38_e6162c2_4days/model.pkl


/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stats for trad-airport-model:sfdps: {'true_positive': 894, 'false_positive': 4, 'false_negative': 6, 'precision': 0.9955456570155902, 'recall': 0.9933333333333333, 'f1': 0.9944382647385984, 'takeoff_airport_ident_match_pct': 0.9809843400447428, 'landing_airport_ident_match_pct': 0.9574944071588367, 'airport_ident_match_pct': 0.9407158836689038}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-07-01_23-01_d3b7b4f_flights_evaluation.csv


In [11]:
from data_engineering.flights.flight_type import add_flight_id_col


df_mismatched_flights = df_comp.filter((pl.col("match_status") == "both") & (pl.col("same_takeoff_airport_ident") == False))
df_mismatched_flights.select(["icao", "takeoff_airport_ident", "landing_airport_ident", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "same_takeoff_airport_ident", "same_landing_airport_ident", "match_status"])
mismatched_flights_ids = add_flight_id_col(df_mismatched_flights).get_column("flight_id")

In [12]:
df.filter(pl.col("flight_id").is_in(mismatched_flights_ids.implode()))

flight_id,airport_ident,score,distance_km,elevation_above_airport_ft,bearing_airport_to_first_deg,bearing_first_to_airport_deg,takeoff_track_error_deg,airport_behind_error_deg,along_track_from_airport_km,cross_track_from_airport_km,candidate_is_behind,rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""a0516f_2026-03-15_17-26""","""US-3265""",0.998133,8.401352,2400.0,0.892688,180.893338,179.293338,179.292688,-8.400713,0.103616,0.0,1
"""a0516f_2026-03-15_17-26""","""XS47""",0.987282,8.296751,1953.0,154.559601,334.577337,27.022663,27.040399,7.390969,3.76957,1.0,2
"""a0516f_2026-03-15_17-26""","""KPEZ""",0.947961,2.763894,1970.0,210.649273,30.642266,29.042266,29.049273,2.416367,1.341745,1.0,3
"""a0516f_2026-03-15_17-26""","""US-5034""",0.442143,13.916355,2048.0,302.812529,122.754447,121.154447,121.212529,-7.199582,11.90928,0.0,4
"""a0516f_2026-03-15_17-26""","""US-0051""",0.114327,19.848794,1900.0,203.875991,23.835949,22.235949,22.275991,18.372706,7.511213,1.0,5
…,…,…,…,…,…,…,…,…,…,…,…,…
"""ab63bc_2026-03-15_17-26""","""2AR7""",0.000145,149.100108,9609.0,55.019126,235.770056,147.470053,146.719123,-125.707867,80.177144,0.0,257
"""ab63bc_2026-03-15_17-26""","""41AR""",0.000135,123.201582,9549.0,63.565771,244.246276,155.946273,155.265768,-112.503207,50.216115,0.0,258
"""ab63bc_2026-03-15_17-26""","""AR05""",0.000132,125.417388,8280.0,122.119369,302.783692,145.516311,146.180634,-103.379973,71.007763,0.0,259
